# Gaussian Mixture Models: The EM Algorithm in Practice

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/unsupervised/gaussian_mixture_models.ipynb)

**Companion blog post:** [Gaussian Mixture Models: The EM Algorithm in Practice](https://sesen.ai/blog/gaussian-mixture-models-em-in-practice)

---

In this notebook, you'll implement a Gaussian Mixture Model (GMM) from scratch using the EM algorithm.

You'll learn:
- How GMMs model data as a mixture of Gaussians
- What responsibilities (soft assignments) are and how the E-step computes them
- How the M-step uses weighted MLE to update parameters
- How to choose the number of components using BIC
- How to compare your implementation with scikit-learn

## 1. The Data: Old Faithful Geyser

Old Faithful is a geyser in Yellowstone National Park. Its eruptions have a clear bimodal pattern: short eruptions (~2 min) and long eruptions (~4.5 min).

Let's load and visualise the data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Old Faithful eruption durations (minutes)
eruptions = np.array([
    3.6, 1.8, 3.333, 2.283, 4.533, 2.883, 4.7, 3.6, 1.95, 4.35,
    1.833, 3.917, 4.2, 1.75, 4.7, 2.167, 1.75, 4.8, 1.6, 4.25,
    1.8, 1.75, 3.45, 3.067, 4.533, 3.6, 1.967, 4.083, 3.85, 4.433,
    4.3, 4.467, 3.367, 4.033, 3.833, 2.017, 1.867, 4.833, 1.833, 4.783,
    4.35, 1.883, 4.567, 1.75, 4.533, 3.317, 3.833, 2.1, 4.633, 2.0,
    4.8, 4.716, 1.833, 4.833, 1.733, 4.883, 3.717, 1.667, 4.567, 4.317,
    2.233, 4.5, 1.75, 4.8, 1.817, 4.4, 4.167, 4.7, 2.067, 4.7,
    4.033, 1.967, 4.5, 4.0, 1.983, 5.067, 2.017, 4.567, 3.883, 3.6,
    4.133, 4.333, 4.1, 2.633, 4.067, 4.933, 3.95, 4.517, 2.167, 4.0,
    2.2, 4.333, 1.867, 4.817, 1.833, 4.3, 4.667, 3.75, 1.867, 4.9,
    2.483, 4.367, 2.1, 4.5, 4.05, 1.867, 4.583, 1.883, 4.367, 1.75,
    4.417, 1.967, 4.185, 1.883, 4.3, 1.767, 4.533, 2.35, 4.533, 4.45,
    3.567, 4.5, 4.15, 3.817, 3.917, 4.45, 2.0, 4.283, 4.767, 4.533,
    1.85, 4.25, 1.983, 2.25, 4.75, 4.117, 2.15, 4.417, 1.817, 4.467,
])

plt.figure(figsize=(10, 4))
plt.hist(eruptions, bins=20, density=True, alpha=0.6, color='steelblue', edgecolor='white')
plt.xlabel('Eruption Duration (minutes)')
plt.ylabel('Density')
plt.title('Old Faithful Eruption Durations — Bimodal Distribution')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Number of eruptions: {len(eruptions)}")
print(f"Range: {eruptions.min():.2f} to {eruptions.max():.2f} minutes")

## 2. Implement the GMM

A GMM assumes data comes from a mixture of K Gaussian distributions:

$$p(x) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(x \mid \mu_k, \sigma_k^2)$$

We need to estimate:
- **Means** μ₁, ..., μₖ — centres of each Gaussian
- **Variances** σ₁², ..., σₖ² — spread of each Gaussian
- **Mixing weights** π₁, ..., πₖ — probability of drawing from each component

In [ ]:
def gaussian_pdf(x, mu, sigma2):
    """Univariate Gaussian probability density function."""
    return (1 / np.sqrt(2 * np.pi * sigma2)) * np.exp(-0.5 * (x - mu)**2 / sigma2)


def fit_gmm(data, K=2, max_iter=50, tol=1e-6, seed=42):
    """
    Fit a Gaussian Mixture Model using EM.

    Args:
        data: 1D array of observations
        K: number of mixture components
        max_iter: maximum EM iterations
        tol: convergence threshold
        seed: random seed for initialisation

    Returns:
        means, variances, weights, responsibilities, log_likelihoods
    """
    N = len(data)

    # Initialise parameters
    rng = np.random.default_rng(seed)
    means = rng.choice(data, K, replace=False)
    variances = np.full(K, np.var(data))
    weights = np.full(K, 1.0 / K)

    log_liks = []

    for iteration in range(max_iter):
        # E-STEP: Compute responsibilities
        resp = np.zeros((N, K))
        for k in range(K):
            resp[:, k] = weights[k] * gaussian_pdf(data, means[k], variances[k])
        resp /= resp.sum(axis=1, keepdims=True)

        # M-STEP: Update parameters
        Nk = resp.sum(axis=0)  # Effective number of points per component
        means = (resp * data[:, None]).sum(axis=0) / Nk
        variances = (resp * (data[:, None] - means)**2).sum(axis=0) / Nk
        weights = Nk / N

        # Compute log-likelihood
        ll = np.sum(np.log(sum(
            weights[k] * gaussian_pdf(data, means[k], variances[k])
            for k in range(K)
        )))
        log_liks.append(ll)

        if iteration > 0 and abs(log_liks[-1] - log_liks[-2]) < tol:
            break

    return means, variances, weights, resp, log_liks


# Fit a 2-component GMM
means, variances, weights, resp, log_liks = fit_gmm(eruptions)

# Sort by mean for consistent display
order = np.argsort(means)
means, variances, weights = means[order], variances[order], weights[order]
resp = resp[:, order]

print(f"Converged in {len(log_liks)} iterations\n")
print(f"Component 1 (short eruptions):")
print(f"  μ = {means[0]:.3f} min, σ = {np.sqrt(variances[0]):.3f}, weight = {weights[0]:.3f}")
print(f"\nComponent 2 (long eruptions):")
print(f"  μ = {means[1]:.3f} min, σ = {np.sqrt(variances[1]):.3f}, weight = {weights[1]:.3f}")

## 3. Visualise the Fit

Let's overlay the fitted Gaussians on the histogram and colour-code points by their responsibilities.

In [ ]:
x_plot = np.linspace(0.5, 6, 300)
mixture = sum(weights[k] * gaussian_pdf(x_plot, means[k], variances[k]) for k in range(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Mixture fit
axes[0].hist(eruptions, bins=20, density=True, alpha=0.4, color='steelblue', edgecolor='white')
colors = ['#e74c3c', '#3498db']
for k in range(2):
    component = weights[k] * gaussian_pdf(x_plot, means[k], variances[k])
    axes[0].plot(x_plot, component, '--', color=colors[k], linewidth=2,
                 label=f'Component {k+1}: μ={means[k]:.2f}')
axes[0].plot(x_plot, mixture, 'k-', linewidth=2, label='Mixture')
axes[0].set_xlabel('Eruption Duration (minutes)')
axes[0].set_ylabel('Density')
axes[0].set_title('GMM Fit to Old Faithful')
axes[0].legend()

# Right: Responsibilities
scatter_colors = resp[:, 1]  # Probability of belonging to long-eruption cluster
axes[1].scatter(range(len(eruptions)), eruptions, c=scatter_colors, cmap='RdBu',
                edgecolors='grey', linewidth=0.5, alpha=0.8)
axes[1].set_xlabel('Observation Index')
axes[1].set_ylabel('Eruption Duration (minutes)')
axes[1].set_title('Soft Assignments (blue = long, red = short)')
cbar = plt.colorbar(axes[1].collections[0], ax=axes[1])
cbar.set_label('P(long eruption)')

plt.tight_layout()
plt.show()

## 4. Trace Through the E-Step

Let's examine the responsibilities for specific data points to build intuition:

In [ ]:
print("Responsibilities (probability of belonging to each cluster):")
print(f"{'Duration':>10s}  {'P(short)':>10s}  {'P(long)':>10s}  {'Assignment':>12s}")
print("-" * 50)

# Pick points from different regions
for idx in [1, 13, 23, 0, 4]:
    dur = eruptions[idx]
    r_short, r_long = resp[idx]
    label = "Short" if r_short > 0.5 else "Long"
    confidence = max(r_short, r_long)
    print(f"{dur:10.2f}  {r_short:10.4f}  {r_long:10.4f}  {label:>8s} ({confidence:.0%})")

print("\nPoints near the boundary (2.5-3.5 min) have more uncertain assignments.")

## 5. Convergence

EM is guaranteed to increase the log-likelihood at every iteration. Let's verify:

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(log_liks) + 1), log_liks, 'g-o', markersize=6)
plt.xlabel('Iteration')
plt.ylabel('Log-Likelihood')
plt.title('EM Monotonically Increases the Log-Likelihood')
plt.grid(True, alpha=0.3)
plt.show()

print("Log-likelihood progression:")
for i, ll in enumerate(log_liks):
    change = f"(+{ll - log_liks[i-1]:.4f})" if i > 0 else ""
    print(f"  Iteration {i+1}: {ll:.4f} {change}")

## 6. Model Selection with BIC

How do we choose the number of components K? The Bayesian Information Criterion (BIC) balances fit against complexity:

$$\text{BIC} = -2\ell(\hat{\theta}) + p \log N$$

Lower is better.

In [ ]:
K_range = range(1, 6)
bics = []

for K in K_range:
    m, v, w, _, lls = fit_gmm(eruptions, K=K)
    n_params = 3 * K - 1  # K means + K variances + (K-1) weights
    bic = -2 * lls[-1] + n_params * np.log(len(eruptions))
    bics.append(bic)

plt.figure(figsize=(8, 4))
plt.bar(list(K_range), bics, color='steelblue', alpha=0.7)
plt.xlabel('Number of Components (K)')
plt.ylabel('BIC (lower is better)')
plt.title('Model Selection: BIC vs Number of Components')
plt.grid(True, alpha=0.3, axis='y')
plt.show()

best_K = list(K_range)[np.argmin(bics)]
print(f"BIC scores:")
for k, bic in zip(K_range, bics):
    marker = " ← best" if k == best_K else ""
    print(f"  K={k}: BIC = {bic:.1f}{marker}")

## 7. Initialisation Sensitivity

Like all EM algorithms, GMMs can get stuck in local optima. Let's see how different seeds affect the result:

In [ ]:
print(f"{'Seed':>6s}  {'μ₁':>8s}  {'μ₂':>8s}  {'Final LL':>12s}  {'Iters':>6s}")
print("-" * 50)

best_ll = -np.inf
best_result = None

for seed in range(10):
    m, v, w, r, lls = fit_gmm(eruptions, seed=seed)
    sorted_means = np.sort(m)
    marker = ""
    if lls[-1] > best_ll:
        best_ll = lls[-1]
        best_result = (m, v, w)
        marker = " ← best"
    print(f"{seed:6d}  {sorted_means[0]:8.3f}  {sorted_means[1]:8.3f}  {lls[-1]:12.4f}  {len(lls):6d}{marker}")

print(f"\nBest log-likelihood: {best_ll:.4f}")

## 8. Compare with scikit-learn

Let's verify our implementation matches the production-ready scikit-learn version:

In [ ]:
from sklearn.mixture import GaussianMixture

# Fit scikit-learn GMM
sklearn_gmm = GaussianMixture(n_components=2, random_state=42)
sklearn_gmm.fit(eruptions.reshape(-1, 1))

sk_means = sklearn_gmm.means_.flatten()
sk_vars = sklearn_gmm.covariances_.flatten()
sk_weights = sklearn_gmm.weights_

# Sort both by mean for comparison
sk_order = np.argsort(sk_means)
our_order = np.argsort(means)

print(f"{'':>15s}  {'Our GMM':>12s}  {'sklearn':>12s}")
print("-" * 45)
for i, label in enumerate(['μ (short)', 'μ (long)']):
    oi, si = our_order[i], sk_order[i]
    print(f"{label:>15s}  {means[oi]:12.4f}  {sk_means[si]:12.4f}")
for i, label in enumerate(['σ (short)', 'σ (long)']):
    oi, si = our_order[i], sk_order[i]
    print(f"{label:>15s}  {np.sqrt(variances[oi]):12.4f}  {np.sqrt(sk_vars[si]):12.4f}")
for i, label in enumerate(['π (short)', 'π (long)']):
    oi, si = our_order[i], sk_order[i]
    print(f"{label:>15s}  {weights[oi]:12.4f}  {sk_weights[si]:12.4f}")

## 9. Exercises

Try these to deepen your understanding:

In [ ]:
# Exercise 1: 2D Clustering
# The Old Faithful dataset also has waiting times between eruptions.
# Extend the GMM to cluster 2D data (duration, waiting).
#
# Hint: Replace gaussian_pdf with a 2D version using np.linalg
# The M-step needs to compute covariance matrices instead of scalar variances.

# Waiting times corresponding to the eruption durations above
waiting = np.array([
    79, 54, 74, 62, 85, 55, 88, 85, 51, 85,
    54, 84, 78, 47, 83, 52, 62, 84, 52, 79,
    51, 47, 78, 69, 74, 83, 55, 76, 78, 79,
    73, 77, 66, 80, 74, 52, 48, 80, 59, 90,
    80, 58, 84, 58, 73, 83, 64, 53, 82, 59,
    75, 90, 54, 80, 54, 83, 71, 64, 77, 81,
    59, 84, 48, 82, 60, 92, 78, 78, 65, 73,
    82, 56, 79, 71, 62, 76, 60, 78, 76, 83,
    75, 82, 70, 65, 73, 88, 76, 80, 48, 86,
    60, 90, 50, 78, 63, 72, 84, 75, 51, 82,
    62, 88, 49, 83, 81, 47, 84, 52, 86, 81,
    75, 59, 89, 45, 93, 72, 71, 54, 79, 77,
    64, 83, 72, 68, 75, 76, 52, 63, 81, 78,
    56, 78, 64, 42, 82, 73, 49, 81, 60, 84,
])

# Your code here:
# data_2d = np.column_stack([eruptions, waiting])
# ...

In [ ]:
# Exercise 2: Anomaly Detection
# Use the fitted GMM to compute p(x) for each data point.
# Flag points with p(x) below the 5th percentile as anomalies.
#
# Hint: p(x) = sum_k pi_k * N(x | mu_k, sigma_k^2)

# Your code here:
# densities = ...
# threshold = np.percentile(densities, 5)
# anomalies = eruptions[densities < threshold]

In [ ]:
# Exercise 3: Generate Synthetic Data
# Since a GMM is a generative model, you can sample from it!
# Generate 500 synthetic eruption durations from your fitted GMM.
#
# Steps:
# 1. For each sample, pick a component k with probability pi_k
# 2. Draw from N(mu_k, sigma_k^2)
# 3. Plot histogram of synthetic vs real data

# Your code here:

In [ ]:
# Exercise 4: Variance Floor
# What happens if you initialise with K=3 and one component
# collapses onto a single point? Add a variance floor (minimum
# allowed variance) to prevent singularities.
#
# Modify fit_gmm to add: variances = np.maximum(variances, 1e-6)
# after the M-step.

# Your code here:

In [ ]:
# Exercise 5: EM Animation
# Create a step-by-step animation showing how the Gaussians
# move and reshape across EM iterations.
#
# Hint: Store the parameters at each iteration and plot them
# in a loop or use matplotlib.animation.

# Your code here:

## 10. Summary

### Key Takeaways

1. **GMMs = soft K-Means** — each point gets a probability of belonging to each cluster, not a hard label
2. **E-step computes responsibilities** — "how much does each component claim each point?"
3. **M-step is weighted MLE** — update means, variances, and weights using responsibilities
4. **EM always increases the log-likelihood** — guaranteed convergence to a local optimum
5. **Use BIC for model selection** — balances goodness of fit vs model complexity

### What's Next?

- **EM Coin Toss** — If you want the gentler introduction: [EM Algorithm Tutorial](https://sesen.ai/blog/em-algorithm-coin-toss-intuitive-guide)
- **MLE Foundations** — Understand the likelihood theory behind the M-step: [MLE from Scratch](https://sesen.ai/blog/maximum-likelihood-estimation-from-scratch)
- **Bayesian GMMs** — Place priors on parameters for automatic model selection: [MCMC Tutorial](https://sesen.ai/blog/mcmc-metropolis-hastings-island-hopping-guide)

---

**Author:** Dr. Berkan Sesen | [sesen.ai](https://sesen.ai)

**Companion blog post:** [Gaussian Mixture Models: The EM Algorithm in Practice](https://sesen.ai/blog/gaussian-mixture-models-em-in-practice)